# 06 — Model selection is not threshold selection

**Objectives**

- Distinguish ranking quality from a binary action rule.
- Reproduce the validation-only cost/constraint threshold search.
- Explain calibration, discrimination, and thresholded decisions separately.

**Prerequisite:** lesson 05 created selection evidence.


In [ ]:
import pandas as pd

import mlflow.sklearn
from aai_local_classification.contracts import SplitName
from aai_local_classification.data import load_split
from aai_local_classification.evaluation import evaluate_probabilities
from aai_local_classification.learning import state_exists
from aai_local_classification.modeling import feature_frame
from aai_local_classification.policy import selection_policy_sha256
from aai_local_classification.tracking import configure_mlflow
from aai_local_classification.workflow import (
    ensure_prepared,
    load_selection,
    run_candidate_selection,
)
from aai_local_classification.learning import study_root
from aai_local_classification.settings import load_settings

settings = load_settings()
root = study_root()
print(f"Course state: {root}")
print(f"Experiment: {settings.experiment_name}")


In [ ]:
paths = configure_mlflow(settings, root)
manifest = ensure_prepared(settings, root)
if not state_exists("selection.json"):
    run_candidate_selection(settings, root)
selection = load_selection(root)
if (
    selection.dataset_sha256 != manifest.dataset_sha256
    or selection.selection_policy_sha256 != selection_policy_sha256(settings)
):
    selection = run_candidate_selection(settings, root)
selected = next(
    item for item in selection.candidates if item.run_id == selection.selected_run_id
)
model = mlflow.sklearn.load_model(selection.selected_model_uri)
validation = load_split(settings, SplitName.VALIDATION, paths.data_root)
x_validation = feature_frame(validation, settings)
y_validation = validation[settings.data.target_column]
probability = model.predict_proba(x_validation)[:, list(model.classes_).index(1)]


In [ ]:
threshold_rows = []
chosen = selected.threshold_selection
thresholds = sorted(
    {
        round(max(0.01, min(0.99, chosen.threshold + offset)), 3)
        for offset in (-0.04, -0.02, 0.0, 0.02, 0.06, 0.12)
    }
)
for threshold in thresholds:
    metrics = evaluate_probabilities(
        y_validation,
        probability,
        threshold,
        false_negative_cost=settings.selection.false_negative_cost,
        false_positive_cost=settings.selection.false_positive_cost,
    )
    threshold_rows.append(
        {
            "threshold": threshold,
            "precision": metrics.precision,
            "recall": metrics.recall,
            "predicted_positive_rate": metrics.predicted_positive_rate,
            "cost_per_1000": metrics.cost_per_1000,
            "average_precision": metrics.average_precision,
        }
    )
pd.DataFrame(threshold_rows)


In [ ]:
pd.Series(
    {
        "selected_candidate": selected.candidate_name,
        "threshold": chosen.threshold,
        "feasible_thresholds": chosen.feasible_threshold_count,
        "validation_precision": chosen.validation_metrics.precision,
        "validation_recall": chosen.validation_metrics.recall,
        "validation_cost_per_1000": chosen.validation_metrics.cost_per_1000,
        "selection_rule": chosen.selection_rule,
    }
).to_frame("value")


Average precision and ROC-AUC stay constant as the threshold changes because
they assess ranking. Precision, recall, review volume, and action cost change.
Brier score/log loss assess probability quality and are also threshold-free.
If no threshold satisfies both declared operating constraints, selection stops
as inconclusive; it never silently drops a requirement.

### Exercise

Change only the false-negative cost in a scratch copy of the settings and
predict which direction the chosen threshold should move.

**Hint:** making missed churn more expensive usually favors a lower threshold
and higher recall, at the price of more reviews.

**Checkpoint:** the candidate and threshold are fixed using validation evidence.
No final-test metric has influenced either choice.

Next: **07_frozen_test_gate.ipynb**.
